In [2]:
"""
Whisper Audio Transcription Script for Google Colab
Optimized for Russian language films with complex audio (background noise, music, nature sounds)

This notebook processes WAV audio files and generates SRT subtitle files
using OpenAI's Whisper large-v3 model via faster-whisper library.

Project structure in Google Drive:
  whisper_project/
    ├─ whisper_transcribe.ipynb  (this notebook)
    ├─ audio/                     (input WAV files)
    ├─ subtitles/                 (output SRT files)
    └─ models/                    (cached Whisper models)

Model: faster-whisper large-v3
Language: Russian
GPU: T4 (required)
Format: SRT subtitles

Author: [Your name]
Created: 2025-11-20
"""

import os
from pathlib import Path

# Expected project path in Google Drive
BASE_PATH = "/content/drive/MyDrive/whisper_project"

# Check if Drive is already mounted
drive_mounted = os.path.exists("/content/drive/MyDrive")

if not drive_mounted:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Drive mounted")
else:
    print("✓ Drive already mounted")

# Verify project structure
print("\n" + "="*70)
print("CHECKING PROJECT STRUCTURE")
print("="*70)

if os.path.exists(BASE_PATH):
    print(f"✓ Project directory found: {BASE_PATH}")

    # Show contents
    print(f"\nProject contents:")
    !ls -lh "$BASE_PATH"

    # Check subdirectories
    audio_dir = Path(f"{BASE_PATH}/audio")
    subtitles_dir = Path(f"{BASE_PATH}/subtitles")

    # Audio directory
    if audio_dir.exists():
        wav_files = list(audio_dir.glob('*.wav')) + list(audio_dir.glob('*.WAV'))
        print(f"\n✓ Audio directory: {len(wav_files)} WAV file(s)")
        if wav_files:
            for f in sorted(wav_files)[:5]:
                print(f"  - {f.name}")
            if len(wav_files) > 5:
                print(f"  ... and {len(wav_files) - 5} more")
        else:
            print("  ⚠️  No WAV files found - upload files to continue")
    else:
        print(f"\n⚠️  Audio directory missing - creating...")
        audio_dir.mkdir(parents=True, exist_ok=True)
        print(f"  ✓ Created: {audio_dir}")

    # Subtitles directory
    if not subtitles_dir.exists():
        print(f"\n  Creating subtitles directory...")
        subtitles_dir.mkdir(parents=True, exist_ok=True)
        print(f"  ✓ Created: {subtitles_dir}")
    else:
        print(f"✓ Subtitles directory exists")

else:
    print(f"✗ ERROR: Project directory not found!")
    print(f"\nExpected path: {BASE_PATH}")
    print(f"\nPlease create this folder structure in Google Drive:")
    print(f"  1. Go to https://drive.google.com")
    print(f"  2. Navigate to 'Мой диск' (My Drive)")
    print(f"  3. Create folder: whisper_project")
    print(f"  4. Inside whisper_project create:")
    print(f"     - audio/      (for input WAV files)")
    print(f"     - subtitles/  (for output SRT files)")
    print(f"\nSearching for similar folders...")
    !find /content/drive/MyDrive -maxdepth 2 -name "*whisper*" -type d 2>/dev/null || echo "  No matches found"

print("="*70)

Mounting Google Drive...
Mounted at /content/drive
✓ Drive mounted

CHECKING PROJECT STRUCTURE
✓ Project directory found: /content/drive/MyDrive/whisper_project

Project contents:
total 46K
drwx------ 2 root root 4.0K Nov 20 17:42 audio
drwx------ 2 root root 4.0K Nov 20 17:43 logs
drwx------ 2 root root 4.0K Nov 20 17:58 models
drwx------ 2 root root 4.0K Nov 20 17:43 subtitles
-rw------- 1 root root  30K Nov 28 16:39 whisper_audio_to_srt.ipynb

✓ Audio directory: 4 WAV file(s)
  - 01.Рикошет.(2019).WEBRip.Files-x.wav
  - 02.Рикошет.(2019).WEBRip.Files-x.wav
  - 03.Рикошет.(2019).WEBRip.Files-x.wav
  - 04.Рикошет.(2019).WEBRip.Files-x.wav
✓ Subtitles directory exists


In [3]:
"""
Install required packages:
- faster-whisper: Optimized Whisper implementation (4-5x faster)
- Uses CTranslate2 for efficient inference on GPU
"""

print("Installing faster-whisper and dependencies...")
print("This may take 2-3 minutes on first run.\n")

!pip install -q faster-whisper

print("\n✓ Installation completed!")

# Suppress HuggingFace token warning
import warnings
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

# Verify GPU availability
print("\nGPU Status:")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

import torch
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: GPU not detected! Processing will be very slow.")
    print("Go to Runtime → Change runtime type → T4 GPU")

Installing faster-whisper and dependencies...
This may take 2-3 minutes on first run.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.0 MB/s eta 0:00:00

✓ Installation completed!

GPU Status:
Tesla T4, 15360 MiB, 15095 MiB

PyTorch CUDA available: True
CUDA device: Tesla T4


In [4]:
"""
Import libraries and configure logging
"""

import os
import sys
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Tuple
from faster_whisper import WhisperModel

# ==================== CONFIGURATION ====================

# Project paths
BASE_PATH = "/content/drive/MyDrive/whisper_project"
AUDIO_DIR = os.path.join(BASE_PATH, "audio")
SUBTITLES_DIR = os.path.join(BASE_PATH, "subtitles")
LOGS_DIR = os.path.join(BASE_PATH, "logs")

# Whisper settings
MODEL_SIZE = "large-v3"
LANGUAGE = "ru"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"

# Processing settings
SKIP_EXISTING = True

# Transcription parameters
BEAM_SIZE = 5
TEMPERATURE = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
CONDITION_ON_PREVIOUS_TEXT = False

# ==================== LOGGING SETUP ====================

def setup_logging() -> logging.Logger:
    """
    Configure logging to both file and console.

    Returns:
        Configured logger instance
    """
    # Create logs directory
    os.makedirs(LOGS_DIR, exist_ok=True)

    # Log filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(LOGS_DIR, f"transcribe_{timestamp}.log")

    # Setup logger
    logger = logging.getLogger("WhisperTranscribe")
    logger.setLevel(logging.INFO)

    # Clear existing handlers
    logger.handlers.clear()

    # Log format
    formatter = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # File handler
    file_handler = logging.FileHandler(log_file, encoding='utf-8')
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(formatter)

    # Console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    # Add handlers
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    logger.info(f"Log file: {log_file}")

    return logger

# Initialize logger
logger = setup_logging()

logger.info("=" * 70)
logger.info("Whisper Audio Transcription Script")
logger.info("=" * 70)
logger.info(f"Model: {MODEL_SIZE}")
logger.info(f"Language: {LANGUAGE}")
logger.info(f"Device: {DEVICE}")
logger.info(f"Beam size: {BEAM_SIZE}")
logger.info(f"Temperature: {TEMPERATURE}")
logger.info(f"Condition on previous text: {CONDITION_ON_PREVIOUS_TEXT}")
logger.info(f"Audio directory: {AUDIO_DIR}")
logger.info(f"Output directory: {SUBTITLES_DIR}")
logger.info("=" * 70)

# Create output directory
os.makedirs(SUBTITLES_DIR, exist_ok=True)

print("\n✓ Configuration loaded successfully!")

2025-11-28 16:40:05 - INFO - Log file: /content/drive/MyDrive/whisper_project/logs/transcribe_20251128_164005.log


INFO:WhisperTranscribe:Log file: /content/drive/MyDrive/whisper_project/logs/transcribe_20251128_164005.log


2025-11-28 16:40:05 - INFO - ======================================================================


INFO:WhisperTranscribe:======================================================================


2025-11-28 16:40:05 - INFO - Whisper Audio Transcription Script


...

2025-11-28 16:40:05 - INFO - Output directory: /content/drive/MyDrive/whisper_project/subtitles


INFO:WhisperTranscribe:Output directory: /content/drive/MyDrive/whisper_project/subtitles


2025-11-28 16:40:05 - INFO - ======================================================================


INFO:WhisperTranscribe:======================================================================



✓ Configuration loaded successfully!


In [5]:
"""
Core transcription functions
"""

def format_timestamp(seconds: float) -> str:
    """
    Convert seconds to SRT timestamp format (HH:MM:SS,mmm).

    Args:
        seconds: Time in seconds

    Returns:
        Formatted timestamp string
    """
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds % 1) * 1000)

    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"


def segments_to_srt(segments: list, output_path: str) -> None:
    """
    Convert Whisper segments to SRT subtitle format with UTF-8 BOM.

    Args:
        segments: List of transcription segments from Whisper
        output_path: Path to save SRT file
    """
    with open(output_path, 'w', encoding='utf-8-sig') as f:
        for i, segment in enumerate(segments, start=1):
            f.write(f"{i}\n")
            start_time = format_timestamp(segment.start)
            end_time = format_timestamp(segment.end)
            f.write(f"{start_time} --> {end_time}\n")
            f.write(f"{segment.text.strip()}\n")
            f.write("\n")


def transcribe_audio(
    audio_path: str,
    output_path: str,
    model: WhisperModel,
    logger: logging.Logger
) -> None:
    """
    Transcribe single audio file to SRT subtitles.

    Args:
        audio_path: Path to input audio file
        output_path: Path to save SRT file
        model: Loaded Whisper model instance
        logger: Logger instance

    Raises:
        Exception: Any error during transcription
    """
    try:
        logger.info(f"Processing: {os.path.basename(audio_path)}")

        segments, info = model.transcribe(
            audio_path,
            language=LANGUAGE,
            beam_size=BEAM_SIZE,
            temperature=TEMPERATURE,
            vad_filter=True,
            vad_parameters=dict(
                min_silence_duration_ms=1000,
                speech_pad_ms=400
            ),
            condition_on_previous_text=CONDITION_ON_PREVIOUS_TEXT,
            initial_prompt="Это российский фильм. Люди разговаривают на русском языке четко и разборчиво."
        )

        segments_list = list(segments)

        logger.info(f"Detected language: {info.language} (probability: {info.language_probability:.2%})")
        logger.info(f"Duration: {info.duration:.1f}s")
        logger.info(f"Segments: {len(segments_list)}")

        segments_to_srt(segments_list, output_path)

        logger.info(f"✓ Successfully saved: {os.path.basename(output_path)}")

    except Exception as e:
        logger.error(f"✗ Error transcribing {os.path.basename(audio_path)}: {e}")
        raise


def process_all_files(
    audio_dir: str,
    output_dir: str,
    model: WhisperModel,
    logger: logging.Logger
) -> Tuple[int, int]:
    """
    Process all WAV files in directory.

    Args:
        audio_dir: Directory containing audio files
        output_dir: Directory for saving SRT files
        model: Loaded Whisper model
        logger: Logger instance

    Returns:
        Tuple of (processed_count, skipped_count)

    Raises:
        Exception: Stops on any processing error
    """
    audio_files = sorted(Path(audio_dir).glob("*.wav"))

    if not audio_files:
        logger.warning(f"No WAV files found in {audio_dir}")
        return 0, 0

    logger.info(f"Found {len(audio_files)} audio file(s)")
    logger.info("-" * 70)

    processed = 0
    skipped = 0

    for audio_file in audio_files:
        srt_file = Path(output_dir) / f"{audio_file.stem}.srt"

        if SKIP_EXISTING and srt_file.exists():
            logger.info(f"⊘ Skipped (already exists): {srt_file.name}")
            skipped += 1
            continue

        transcribe_audio(
            str(audio_file),
            str(srt_file),
            model,
            logger
        )
        processed += 1

    logger.info("-" * 70)
    logger.info(f"Processing completed!")
    logger.info(f"Processed: {processed}")
    logger.info(f"Skipped: {skipped}")
    logger.info(f"Total: {len(audio_files)}")

    return processed, skipped

print("✓ Functions defined successfully!")

✓ Functions defined successfully!


In [ ]:
"""
Main execution cell - RUN THIS TO START TRANSCRIPTION
"""

# Validate directories
if not os.path.exists(AUDIO_DIR):
    logger.error(f"✗ Audio directory not found: {AUDIO_DIR}")
    logger.error("Please upload WAV files to whisper_project/audio/ in Google Drive")
    raise FileNotFoundError(f"Audio directory not found: {AUDIO_DIR}")

logger.info("Loading Whisper model...")
logger.info(f"Model size: {MODEL_SIZE}")
logger.info("This may take 1-2 minutes on first run (model download ~3GB)")
logger.info("-" * 70)

# Load model
# download_root: cache model in Google Drive to avoid re-downloading
model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    download_root=os.path.join(BASE_PATH, "models")  # Cache in project folder
)

logger.info("✓ Model loaded successfully!")
logger.info("=" * 70)

# Process all files
try:
    start_time = datetime.now()
    logger.info(f"Processing started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    logger.info("=" * 70)

    processed, skipped = process_all_files(
        AUDIO_DIR,
        SUBTITLES_DIR,
        model,
        logger
    )

    end_time = datetime.now()
    duration = end_time - start_time

    logger.info("=" * 70)
    logger.info(f"All processing completed successfully!")
    logger.info(f"Total time: {duration}")
    if processed > 0:
        logger.info(f"Average time per file: {duration / processed}")
    logger.info("=" * 70)

    print("\n" + "🎉" * 35)
    print(f"✓ SUCCESS! Processed {processed} file(s)")
    print(f"✓ Subtitles saved to: {SUBTITLES_DIR}")
    if skipped > 0:
        print(f"  (Skipped {skipped} existing file(s))")
    print("🎉" * 35)

except Exception as e:
    logger.error("=" * 70)
    logger.error(f"✗ CRITICAL ERROR: {e}")
    logger.exception("Traceback:")
    logger.error("=" * 70)
    raise

2025-11-28 16:40:22 - INFO - Loading Whisper model...


INFO:WhisperTranscribe:Loading Whisper model...


2025-11-28 16:40:22 - INFO - Model size: large-v3


INFO:WhisperTranscribe:Model size: large-v3


2025-11-28 16:40:22 - INFO - This may take 1-2 minutes on first run (model download ~3GB)


...

INFO:WhisperTranscribe:Segments: 424


2025-11-28 16:45:58 - INFO - ✓ Successfully saved: 02.Рикошет.(2019).WEBRip.Files-x.srt


INFO:WhisperTranscribe:✓ Successfully saved: 02.Рикошет.(2019).WEBRip.Files-x.srt


2025-11-28 16:45:58 - INFO - Processing: 03.Рикошет.(2019).WEBRip.Files-x.wav


INFO:WhisperTranscribe:Processing: 03.Рикошет.(2019).WEBRip.Files-x.wav
